# LaBSE Fine-tuned Model Evaluation on IN22-Conv

This notebook evaluates:

1. **LaBSE base**
2. **LaBSE fine-tuned on bottom5 directed pairs**
3. **LaBSE fine-tuned on top5 directed pairs**

Evaluation is done on **IN22-Conv**.

The notebook evaluates **all directed language pairs found in IN22-Conv**, excluding same-language pairs.

It also produces special summaries for the requested combinations:

- LaBSE base on bottom5
- Bottom5 fine-tuned on bottom5
- Top5 fine-tuned on bottom5
- LaBSE base on top5
- Bottom5 fine-tuned on top5
- Top5 fine-tuned on top5

Metrics included:

- mean gold cosine ± SD
- mean random cosine ± SD
- cosine gap ± SD
- sensitivity at midpoint threshold
- specificity at midpoint threshold
- balanced accuracy at midpoint threshold
- sensitivity at optimal threshold
- specificity at optimal threshold
- best F1 at optimal threshold

## 1. Install dependencies

Run this cell first. If you are using Colab and it asks you to restart the runtime after installation, restart and continue from the imports cell.

In [ ]:
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    _guard_dir = _candidate / "scripts"
    if (_guard_dir / "import_guard.py").exists():
        if str(_guard_dir) not in sys.path:
            sys.path.insert(0, str(_guard_dir))
        break
else:
    raise RuntimeError("Could not locate scripts/import_guard.py. Run this notebook from the WSAI workspace or copy the guard module alongside it.")

from import_guard import install_pandas_guards
install_pandas_guards()


In [ ]:
%pip -q install -U \
  "sentence-transformers" \
  "datasets" \
  "accelerate" \
  "transformers>=4.51.0,<5" \
  "huggingface_hub" \
  "tqdm" \
  "openpyxl" \
  "pandas==2.2.2" \
  "numpy==2.0.2" \
  "scikit-learn>=1.5,<1.9"

## 2. Imports and global setup

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

from pathlib import Path
import zipfile
import shutil
import hashlib
import random
import gc
from itertools import product

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

IS_COLAB = "google.colab" in str(get_ipython())
print("Is Colab:", IS_COLAB)

## 3. Configure paths

Set `ZIP_PATH` to your downloaded zip file.

Examples:

```python
ZIP_PATH = "/content/labse_finetuning_important_results.zip"
ZIP_PATH = "/kaggle/input/my-models/labse_finetuning_important_results.zip"
ZIP_PATH = "/workspace/labse_directed_pair_finetuning/exports/labse_finetuning_important_results.zip"
```

If you already extracted the zip, set `ZIP_PATH = None` and set `EXTRACT_DIR` to the extracted folder.

In [ ]:
BASE_MODEL_NAME = "sentence-transformers/LaBSE"

# Change this to your zip location.
ZIP_PATH = None
# Example:
# ZIP_PATH = "/content/labse_finetuning_important_results.zip"

# Folder where the zip will be extracted, or where it is already extracted.
EXTRACT_DIR = Path("./labse_extracted")

# Evaluation outputs will be saved here.
EVAL_OUTPUT_DIR = Path("./labse_in22_conv_eval_outputs")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 128
EVAL_BATCH_SIZE = 128

# Set True if you want to delete the old extraction folder before extracting.
CLEAR_EXTRACT_DIR_BEFORE_UNZIP = False

print("ZIP_PATH:", ZIP_PATH)
print("EXTRACT_DIR:", EXTRACT_DIR.resolve())
print("EVAL_OUTPUT_DIR:", EVAL_OUTPUT_DIR.resolve())

## 4. Extract zip and find fine-tuned model folders

In [ ]:
def extract_zip_if_needed(zip_path, extract_dir, clear_first=False):
    extract_dir = Path(extract_dir)

    if zip_path is None:
        print("ZIP_PATH is None. Assuming models are already extracted.")
        extract_dir.mkdir(parents=True, exist_ok=True)
        return extract_dir

    zip_path = Path(zip_path)
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP file not found: {zip_path}")

    if clear_first and extract_dir.exists():
        print("Deleting old extracted folder:", extract_dir)
        shutil.rmtree(extract_dir)

    extract_dir.mkdir(parents=True, exist_ok=True)

    print("Extracting:", zip_path)
    print("To:", extract_dir)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    print("Extraction complete.")
    return extract_dir


def find_sentence_transformer_model_dirs(root_dir):
    root_dir = Path(root_dir)
    model_dirs = []

    for p in root_dir.rglob("*"):
        if p.is_dir() and (p / "modules.json").exists():
            model_dirs.append(p)

    return sorted(model_dirs)


EXTRACT_DIR = extract_zip_if_needed(
    ZIP_PATH,
    EXTRACT_DIR,
    clear_first=CLEAR_EXTRACT_DIR_BEFORE_UNZIP
)

model_dirs = find_sentence_transformer_model_dirs(EXTRACT_DIR)

print("Found SentenceTransformer model folders:", len(model_dirs))
for i, p in enumerate(model_dirs):
    print(f"{i:02d} -> {p}")

## 5. Auto-select top5 and bottom5 fine-tuned models

The notebook tries to find:

- `top5_directed_pairs/best_model`
- `bottom5_directed_pairs/best_model`

If auto-selection fails, manually paste the correct paths in `TOP5_BEST_MODEL` and `BOTTOM5_BEST_MODEL`.

In [ ]:
def pick_model_dir(model_dirs, run_keyword, preferred_variant="best_model"):
    model_dirs = [Path(p) for p in model_dirs]

    exact = [
        p for p in model_dirs
        if run_keyword in str(p) and p.name == preferred_variant
    ]
    if exact:
        return exact[0]

    fallback = [
        p for p in model_dirs
        if run_keyword in str(p) and preferred_variant in str(p)
    ]
    if fallback:
        return fallback[0]

    any_match = [
        p for p in model_dirs
        if run_keyword in str(p)
    ]
    if any_match:
        return any_match[0]

    return None


TOP5_BEST_MODEL = pick_model_dir(model_dirs, "top5_directed_pairs", "best_model")
BOTTOM5_BEST_MODEL = pick_model_dir(model_dirs, "bottom5_directed_pairs", "best_model")

print("Auto-selected TOP5_BEST_MODEL:", TOP5_BEST_MODEL)
print("Auto-selected BOTTOM5_BEST_MODEL:", BOTTOM5_BEST_MODEL)

# If auto-selection fails, manually set paths here:
# TOP5_BEST_MODEL = Path("/path/to/top5_directed_pairs/best_model")
# BOTTOM5_BEST_MODEL = Path("/path/to/bottom5_directed_pairs/best_model")

if TOP5_BEST_MODEL is None:
    raise ValueError("Could not auto-find top5 best model. Set TOP5_BEST_MODEL manually.")

if BOTTOM5_BEST_MODEL is None:
    raise ValueError("Could not auto-find bottom5 best model. Set BOTTOM5_BEST_MODEL manually.")

for p in [TOP5_BEST_MODEL, BOTTOM5_BEST_MODEL]:
    if not (Path(p) / "modules.json").exists():
        raise ValueError(f"Invalid SentenceTransformer model folder: {p}")

MODELS_TO_EVALUATE = {
    "labse_base": BASE_MODEL_NAME,
    "labse_bottom5_directed_pairs_best": str(BOTTOM5_BEST_MODEL),
    "labse_top5_directed_pairs_best": str(TOP5_BEST_MODEL),
}

print("\nModels to evaluate:")
for name, path in MODELS_TO_EVALUATE.items():
    print(name, "->", path)

## 6. Define top5 and bottom5 directed-pair groups

In [ ]:
TOP5_DIRECTED_PAIRS = [
    ("kan", "tel"),
    ("urd", "guj"),
    ("ben", "kan"),
    ("guj", "mar"),
    ("guj", "urd"),
]

BOTTOM5_DIRECTED_PAIRS = [
    ("sat", "npi"),
    ("sat", "urd"),
    ("sat", "hin"),
    ("kan", "sat"),
    ("ben", "mni"),
]

TOP5_SET = set(TOP5_DIRECTED_PAIRS)
BOTTOM5_SET = set(BOTTOM5_DIRECTED_PAIRS)

def pair_group(src_short, tgt_short):
    pair = (src_short, tgt_short)
    if pair in TOP5_SET:
        return "top5"
    if pair in BOTTOM5_SET:
        return "bottom5"
    return "other"

def pair_label(src_short, tgt_short):
    return f"{src_short}→{tgt_short}"

print("Top5:", [pair_label(*p) for p in TOP5_DIRECTED_PAIRS])
print("Bottom5:", [pair_label(*p) for p in BOTTOM5_DIRECTED_PAIRS])

## 7. Load IN22-Conv

In [ ]:
def load_in22_dataset_as_dataframe(
    dataset_name,
    dataset_config="default",
    preferred_splits=("conv", "test", "validation", "train")
):
    try:
        if dataset_config is None or dataset_config == "":
            dataset_obj = load_dataset(dataset_name)
        else:
            dataset_obj = load_dataset(dataset_name, dataset_config)

    except Exception as e:
        raise RuntimeError(
            f"Could not load {dataset_name} with config={dataset_config!r}. "
            f"Original error: {e}"
        ) from e

    if hasattr(dataset_obj, "keys"):
        available_splits = list(dataset_obj.keys())
        chosen_split = None

        for split in preferred_splits:
            if split in available_splits:
                chosen_split = split
                break

        if chosen_split is None:
            chosen_split = available_splits[0]

        print(f"Available splits for {dataset_name}: {available_splits}")
        print(f"Using split for {dataset_name}: {chosen_split}")

        return dataset_obj[chosen_split].to_pandas()

    return dataset_obj.to_pandas()


conv_df_raw = load_in22_dataset_as_dataframe(
    "ai4bharat/IN22-Conv",
    "default",
    preferred_splits=("conv", "test", "validation", "train")
)

print("Raw IN22-Conv shape:", conv_df_raw.shape)
print("Columns:")
print(conv_df_raw.columns.tolist())
conv_df_raw.head()

## 8. Detect language columns and build all directed pairs

The notebook evaluates every directed pair:

```text
source language ≠ target language
```

If IN22-Conv has 23 language columns, this gives:

```text
23 × 22 = 506 directed pairs
```

In [ ]:
import re

METADATA_COLS = {
    "context", "source", "url", "domain", "num_words", "bucket", "id"
}

def is_language_column(col):
    # Examples: hin_Deva, eng_Latn, tam_Taml, snd_Deva
    return bool(re.match(r"^[a-z]{3}_[A-Za-z]{4}$", col))

language_cols = [
    col for col in conv_df_raw.columns
    if is_language_column(col)
]

if not language_cols:
    language_cols = [
        col for col in conv_df_raw.columns
        if col not in METADATA_COLS
    ]

language_cols = sorted(language_cols)

LANG_SHORT_TO_COL = {col.split("_")[0]: col for col in language_cols}
LANG_COL_TO_SHORT = {col: col.split("_")[0] for col in language_cols}

eval_langs = sorted(LANG_SHORT_TO_COL.keys())

print("Detected language columns:", len(language_cols))
for short in eval_langs:
    print(f"{short:4s} -> {LANG_SHORT_TO_COL[short]}")

ALL_DIRECTED_PAIRS = [
    (src, tgt)
    for src in eval_langs
    for tgt in eval_langs
    if src != tgt
]

print("\nTotal directed pairs:", len(ALL_DIRECTED_PAIRS))
print("First 20 pairs:", [pair_label(*p) for p in ALL_DIRECTED_PAIRS[:20]])

## 9. Clean evaluation dataframe

In [ ]:
conv_df = conv_df_raw.copy()

for col in language_cols:
    conv_df[col] = conv_df[col].astype(str).str.strip()

mask = np.ones(len(conv_df), dtype=bool)
for col in language_cols:
    mask &= conv_df[col].ne("")
    mask &= conv_df[col].str.lower().ne("nan")
    mask &= conv_df[col].notna()

conv_df = conv_df.loc[mask].reset_index(drop=True)

print("Clean IN22-Conv shape:", conv_df.shape)

if len(conv_df) < 2:
    raise ValueError("Need at least 2 aligned rows to create random wrong pairs.")

conv_df[language_cols].head()

## 10. Metric helper functions

Gold pairs are row-aligned:

```text
source row i ↔ target row i
```

Random wrong pairs are created using a deterministic shuffled off-diagonal target index.

In [ ]:
def stable_int_from_key(key, modulo=2**32 - 1):
    digest = hashlib.md5(str(key).encode("utf-8")).hexdigest()
    return int(digest[:12], 16) % modulo


def make_deranged_indices(n, key):
    """
    Creates a deterministic shuffled index array with no fixed points.
    This is used for random wrong pairs.
    """
    if n < 2:
        raise ValueError("Need n >= 2 for off-diagonal random pairs.")

    seed = stable_int_from_key(key)
    rng = np.random.default_rng(seed)

    idx = np.arange(n)

    for _ in range(100):
        perm = rng.permutation(n)
        if np.all(perm != idx):
            return perm

    shift = seed % (n - 1) + 1
    return np.roll(idx, shift)


def compute_threshold_metrics(gold_cos, random_cos, num_grid=200):
    """
    Positive class = gold pairs
    Negative class = random pairs
    """
    gold_cos = np.asarray(gold_cos, dtype=float)
    random_cos = np.asarray(random_cos, dtype=float)

    mean_gold = float(np.mean(gold_cos))
    mean_random = float(np.mean(random_cos))

    sd_gold = float(np.std(gold_cos, ddof=1)) if len(gold_cos) > 1 else 0.0
    sd_random = float(np.std(random_cos, ddof=1)) if len(random_cos) > 1 else 0.0

    gap_values = gold_cos - random_cos
    mean_gap = float(mean_gold - mean_random)
    sd_gap = float(np.std(gap_values, ddof=1)) if len(gap_values) > 1 else 0.0

    threshold_midpoint = (mean_gold + mean_random) / 2.0
    sensitivity_midpoint = float(np.mean(gold_cos >= threshold_midpoint))
    specificity_midpoint = float(np.mean(random_cos < threshold_midpoint))
    balanced_accuracy_midpoint = (sensitivity_midpoint + specificity_midpoint) / 2.0

    all_scores = np.concatenate([gold_cos, random_cos])
    lo = float(np.min(all_scores))
    hi = float(np.max(all_scores))

    if lo == hi:
        thresholds = np.array([lo])
    else:
        thresholds = np.linspace(lo, hi, num_grid)

    best = {
        "threshold": thresholds[0],
        "f1": -1.0,
        "precision": 0.0,
        "sensitivity": 0.0,
        "specificity": 0.0,
        "balanced_accuracy": 0.0,
    }

    for t in thresholds:
        tp = int(np.sum(gold_cos >= t))
        fn = int(np.sum(gold_cos < t))
        fp = int(np.sum(random_cos >= t))
        tn = int(np.sum(random_cos < t))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        f1 = (
            2 * precision * sensitivity / (precision + sensitivity)
            if (precision + sensitivity) > 0
            else 0.0
        )

        balanced_accuracy = (sensitivity + specificity) / 2.0

        if f1 > best["f1"]:
            best = {
                "threshold": float(t),
                "f1": float(f1),
                "precision": float(precision),
                "sensitivity": float(sensitivity),
                "specificity": float(specificity),
                "balanced_accuracy": float(balanced_accuracy),
            }

    return {
        "mean_gold_cosine": mean_gold,
        "sd_gold_cosine": sd_gold,
        "mean_random_cosine": mean_random,
        "sd_random_cosine": sd_random,
        "cosine_gap": mean_gap,
        "sd_cosine_gap": sd_gap,
        "threshold_midpoint": float(threshold_midpoint),
        "sensitivity_midpoint": sensitivity_midpoint,
        "specificity_midpoint": specificity_midpoint,
        "balanced_accuracy_midpoint": float(balanced_accuracy_midpoint),
        "optimal_threshold": best["threshold"],
        "sensitivity_optimal": best["sensitivity"],
        "specificity_optimal": best["specificity"],
        "balanced_accuracy_optimal": best["balanced_accuracy"],
        "precision_optimal": best["precision"],
        "best_f1_optimal": best["f1"],
    }


def fmt_mean_sd(mean, sd, decimals=4):
    return f"{mean:.{decimals}f} ± {sd:.{decimals}f}"

## 11. Model evaluation function

In [ ]:
def encode_all_languages(model, eval_df, lang_short_to_col, batch_size=128):
    embeddings_by_lang = {}

    for short, col in tqdm(lang_short_to_col.items(), desc="Encoding languages"):
        sentences = eval_df[col].astype(str).tolist()

        emb = model.encode(
            sentences,
            batch_size=batch_size,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=True,
        )

        embeddings_by_lang[short] = emb.astype(np.float32)

    return embeddings_by_lang


def evaluate_model_on_all_pairs(model_path_or_name, model_name, eval_df, lang_short_to_col, directed_pairs):
    print("\n" + "#" * 100)
    print("Evaluating model:", model_name)
    print("Model path/name:", model_path_or_name)
    print("#" * 100)

    model = SentenceTransformer(str(model_path_or_name), device=DEVICE)
    model.max_seq_length = MAX_SEQ_LENGTH

    embeddings_by_lang = encode_all_languages(
        model=model,
        eval_df=eval_df,
        lang_short_to_col=lang_short_to_col,
        batch_size=EVAL_BATCH_SIZE,
    )

    records = []
    n = len(eval_df)

    for src, tgt in tqdm(directed_pairs, desc=f"Pairs for {model_name}"):
        src_emb = embeddings_by_lang[src]
        tgt_emb = embeddings_by_lang[tgt]

        gold_cos = np.sum(src_emb * tgt_emb, axis=1)

        rand_idx = make_deranged_indices(
            n,
            key=f"{SEED}|{model_name}|{src}|{tgt}|random_negative"
        )
        random_tgt_emb = tgt_emb[rand_idx]
        random_cos = np.sum(src_emb * random_tgt_emb, axis=1)

        metrics = compute_threshold_metrics(
            gold_cos=gold_cos,
            random_cos=random_cos,
            num_grid=200,
        )

        rec = {
            "model_name": model_name,
            "source_language": src,
            "target_language": tgt,
            "source_column": lang_short_to_col[src],
            "target_column": lang_short_to_col[tgt],
            "directed_pair": pair_label(src, tgt),
            "pair_group": pair_group(src, tgt),
            "n_eval": n,
        }
        rec.update(metrics)

        rec["mean_gold_cosine_pm_sd"] = fmt_mean_sd(
            rec["mean_gold_cosine"],
            rec["sd_gold_cosine"]
        )
        rec["mean_random_cosine_pm_sd"] = fmt_mean_sd(
            rec["mean_random_cosine"],
            rec["sd_random_cosine"]
        )
        rec["cosine_gap_pm_sd"] = fmt_mean_sd(
            rec["cosine_gap"],
            rec["sd_cosine_gap"]
        )

        records.append(rec)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(records)

## 12. Run full IN22-Conv all-pairs evaluation

In [ ]:
all_model_dfs = []

for model_name, model_path_or_name in MODELS_TO_EVALUATE.items():
    model_df = evaluate_model_on_all_pairs(
        model_path_or_name=model_path_or_name,
        model_name=model_name,
        eval_df=conv_df,
        lang_short_to_col=LANG_SHORT_TO_COL,
        directed_pairs=ALL_DIRECTED_PAIRS,
    )

    out_path = EVAL_OUTPUT_DIR / f"in22_conv_all_pairs_{model_name}.csv"
    model_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    all_model_dfs.append(model_df)

all_pairs_df = pd.concat(all_model_dfs, ignore_index=True)

all_pairs_path = EVAL_OUTPUT_DIR / "in22_conv_eval_all_models_all_directed_pairs.csv"
all_pairs_df.to_csv(all_pairs_path, index=False)

print("Saved all-pairs results:", all_pairs_path)
print("Shape:", all_pairs_df.shape)
all_pairs_df.head()

## 13. Summaries by model and pair group

In [ ]:
metric_cols = [
    "mean_gold_cosine",
    "mean_random_cosine",
    "cosine_gap",
    "sensitivity_midpoint",
    "specificity_midpoint",
    "balanced_accuracy_midpoint",
    "sensitivity_optimal",
    "specificity_optimal",
    "balanced_accuracy_optimal",
    "precision_optimal",
    "best_f1_optimal",
]

def summarize_group(df, group_cols):
    summary = (
        df.groupby(group_cols)[metric_cols]
        .mean()
        .reset_index()
    )

    n_pairs = (
        df.groupby(group_cols)["directed_pair"]
        .nunique()
        .reset_index(name="n_directed_pairs")
    )

    summary = summary.merge(n_pairs, on=group_cols, how="left")
    return summary


summary_by_model = summarize_group(all_pairs_df, ["model_name"])
summary_by_group = summarize_group(all_pairs_df, ["model_name", "pair_group"])

summary_by_model_path = EVAL_OUTPUT_DIR / "in22_conv_summary_by_model_all_pairs.csv"
summary_by_group_path = EVAL_OUTPUT_DIR / "in22_conv_summary_by_model_pair_group.csv"

summary_by_model.to_csv(summary_by_model_path, index=False)
summary_by_group.to_csv(summary_by_group_path, index=False)

print("Saved:", summary_by_model_path)
print("Saved:", summary_by_group_path)

display(summary_by_model)
display(summary_by_group)

## 14. Requested six combinations

This table gives exactly the combinations:

- LaBSE base / bottom5
- Bottom5 fine-tuned / bottom5
- Top5 fine-tuned / bottom5
- LaBSE base / top5
- Bottom5 fine-tuned / top5
- Top5 fine-tuned / top5

In [ ]:
requested_models = [
    "labse_base",
    "labse_bottom5_directed_pairs_best",
    "labse_top5_directed_pairs_best",
]

requested_groups = ["bottom5", "top5"]

requested_six = summary_by_group[
    summary_by_group["model_name"].isin(requested_models)
    & summary_by_group["pair_group"].isin(requested_groups)
].copy()

model_display = {
    "labse_base": "LaBSE base",
    "labse_bottom5_directed_pairs_best": "Bottom5 fine-tuned",
    "labse_top5_directed_pairs_best": "Top5 fine-tuned",
}

requested_six["model_display"] = requested_six["model_name"].map(model_display)
requested_six["combination"] = requested_six["model_display"] + " / " + requested_six["pair_group"]

order = [
    ("labse_base", "bottom5"),
    ("labse_bottom5_directed_pairs_best", "bottom5"),
    ("labse_top5_directed_pairs_best", "bottom5"),
    ("labse_base", "top5"),
    ("labse_bottom5_directed_pairs_best", "top5"),
    ("labse_top5_directed_pairs_best", "top5"),
]

order_df = pd.DataFrame(order, columns=["model_name", "pair_group"])
order_df["order"] = range(len(order_df))

requested_six = requested_six.merge(order_df, on=["model_name", "pair_group"], how="left")
requested_six = requested_six.sort_values("order").drop(columns=["order"]).reset_index(drop=True)

requested_six_path = EVAL_OUTPUT_DIR / "in22_conv_requested_six_combinations_summary.csv"
requested_six.to_csv(requested_six_path, index=False)

print("Saved:", requested_six_path)
display(requested_six)

## 15. Deltas versus LaBSE base

In [ ]:
def make_delta_vs_base(all_df, tuned_model_name, base_model_name="labse_base"):
    base = all_df[all_df["model_name"] == base_model_name].copy()
    tuned = all_df[all_df["model_name"] == tuned_model_name].copy()

    merge_keys = [
        "source_language",
        "target_language",
        "directed_pair",
        "pair_group",
    ]

    merged = tuned[merge_keys + metric_cols].merge(
        base[merge_keys + metric_cols],
        on=merge_keys,
        suffixes=("_tuned", "_base"),
        how="inner",
    )

    merged.insert(0, "tuned_model_name", tuned_model_name)
    merged.insert(1, "base_model_name", base_model_name)

    for m in metric_cols:
        merged[f"delta_{m}"] = merged[f"{m}_tuned"] - merged[f"{m}_base"]

    return merged


delta_bottom5 = make_delta_vs_base(all_pairs_df, "labse_bottom5_directed_pairs_best")
delta_top5 = make_delta_vs_base(all_pairs_df, "labse_top5_directed_pairs_best")

delta_bottom5_path = EVAL_OUTPUT_DIR / "delta_labse_bottom5_best_vs_labse_base_all_directed_pairs.csv"
delta_top5_path = EVAL_OUTPUT_DIR / "delta_labse_top5_best_vs_labse_base_all_directed_pairs.csv"

delta_bottom5.to_csv(delta_bottom5_path, index=False)
delta_top5.to_csv(delta_top5_path, index=False)

print("Saved:", delta_bottom5_path)
print("Saved:", delta_top5_path)

display(delta_bottom5.head())
display(delta_top5.head())

## 16. Delta summaries by pair group

In [ ]:
delta_metric_cols = [f"delta_{m}" for m in metric_cols]

delta_bottom5_summary = (
    delta_bottom5.groupby(["tuned_model_name", "pair_group"])[delta_metric_cols]
    .mean()
    .reset_index()
)

delta_top5_summary = (
    delta_top5.groupby(["tuned_model_name", "pair_group"])[delta_metric_cols]
    .mean()
    .reset_index()
)

delta_summary = pd.concat([delta_bottom5_summary, delta_top5_summary], ignore_index=True)

delta_summary_path = EVAL_OUTPUT_DIR / "delta_summary_vs_labse_base_by_pair_group.csv"
delta_summary.to_csv(delta_summary_path, index=False)

print("Saved:", delta_summary_path)
display(delta_summary)

## 17. Simple plots

In [ ]:
import matplotlib.pyplot as plt

plot_df = requested_six.copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["combination"], plot_df["cosine_gap"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean cosine gap")
plt.title("IN22-Conv: Cosine Gap for Requested Six Combinations")
plt.tight_layout()
cosine_gap_plot_path = EVAL_OUTPUT_DIR / "requested_six_cosine_gap.png"
plt.savefig(cosine_gap_plot_path, dpi=200)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["combination"], plot_df["balanced_accuracy_midpoint"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Balanced accuracy at midpoint")
plt.title("IN22-Conv: Balanced Accuracy for Requested Six Combinations")
plt.tight_layout()
bal_acc_plot_path = EVAL_OUTPUT_DIR / "requested_six_balanced_accuracy.png"
plt.savefig(bal_acc_plot_path, dpi=200)
plt.show()

print("Saved plots:")
print(cosine_gap_plot_path)
print(bal_acc_plot_path)

## 18. Zip evaluation outputs

In [ ]:
zip_path = EVAL_OUTPUT_DIR / "in22_conv_eval_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in EVAL_OUTPUT_DIR.rglob("*"):
        if file.is_file() and file != zip_path:
            z.write(file, arcname=file.relative_to(EVAL_OUTPUT_DIR))

print("Saved zip:", zip_path)
print("Zip size MB:", round(zip_path.stat().st_size / (1024 ** 2), 2))

## 19. How to interpret the metrics

Use this priority order:

1. **Cosine gap**: primary separation signal. Higher is better.
2. **Random cosine**: should stay low. If this increases too much, the model may be giving high cosine to wrong pairs.
3. **Balanced accuracy**: checks both accepting true pairs and rejecting random pairs.
4. **Specificity**: very important for detecting the “high cosine for everything” problem.
5. **Sensitivity**: checks whether true pairs are above the threshold.
6. **Best F1 optimal**: best possible threshold-based discrimination using 200 grid thresholds.

A fine-tuned model is genuinely better only if gold cosine improves **without** random cosine inflating too much, and especially if cosine gap, specificity, balanced accuracy, and best F1 improve.